# Onion Quality Assessment — Google Colab Training & Local Export
This notebook is designed to run seamlessly from **VS Code via the Google Colab extension**.
It trains:
1. **Model 1**: YOLOv8-seg single-class `onion` detector & segmenter (Target: mAP@0.5 ≥ 85%)
2. **Model 2**: MobileNetV3 4-class defect classifier (`healthy`, `sprouted`, `rotten`, `mechanical_damage`) (Target: F1 ≥ 90%)

At the end, a dedicated cell enables **direct download of weights and evaluation metrics to your local PC without using Colab's Web UI**.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
!pip install -q ultralytics opencv-python-headless pydantic imagehash pandas matplotlib tqdm torchvision

In [ ]:
# 2. Clone or Sync Project Code
# If running via VS Code Colab extension, your workspace files are accessible.
# Otherwise, clone your repo or copy code:
import os
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")
os.makedirs("/content/runs", exist_ok=True)
os.makedirs("/content/exports", exist_ok=True)

In [ ]:
# 3. Train Model 1 (YOLOv8n-seg) — Single Class 'onion'
from ultralytics import YOLO

# Load pretrained YOLOv8-nano segmentation model
model1 = YOLO('yolov8n-seg.pt')

# Train on prepared dataset (data/model1/data.yaml)
# Target: mAP@0.5 >= 85%
results_m1 = model1.train(
    data='/content/data/model1/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='/content/runs/model1',
    name='yolov8_seg_onion',
    save=True
)

# Export to ONNX for fast mobile/handheld inference
exported_m1 = model1.export(format='onnx', imgsz=640, dynamic=True)
print(f"Model 1 exported to: {exported_m1}")

In [ ]:
# 4. Train Model 2 (Defect Classifier) — 4 Core Classes
# Classes: healthy, sprouted, rotten, mechanical_damage
model2 = YOLO('yolov8n-cls.pt')  # Or fine-tune torchvision.models.mobilenet_v3_small

results_m2 = model2.train(
    data='/content/data/model2',
    epochs=40,
    imgsz=224,
    batch=32,
    project='/content/runs/model2',
    name='defect_classifier_4class',
    save=True
)

# Export to ONNX
exported_m2 = model2.export(format='onnx', imgsz=224)
print(f"Model 2 exported to: {exported_m2}")

In [ ]:
# 5. Package Models, Metrics & Evaluation Reports into a Zip Archive
import subprocess
from pathlib import Path

archive_name = "onion_grading_colab_artifacts.zip"
archive_path = f"/content/{archive_name}"

# Copy ONNX models and best PyTorch weights to export directory
!mkdir -p /content/exports/weights /content/exports/reports
!cp /content/runs/model1/yolov8_seg_onion/weights/best.pt /content/exports/weights/model1_yolov8_seg.pt || true
!cp /content/runs/model1/yolov8_seg_onion/weights/best.onnx /content/exports/weights/model1_yolov8_seg.onnx || true
!cp /content/runs/model2/defect_classifier_4class/weights/best.pt /content/exports/weights/model2_defect_cls.pt || true
!cp /content/runs/model2/defect_classifier_4class/weights/best.onnx /content/exports/weights/model2_defect_cls.onnx || true
!cp /content/runs/model1/yolov8_seg_onion/results.csv /content/exports/reports/model1_eval.csv || true
!cp /content/runs/model2/defect_classifier_4class/results.csv /content/exports/reports/model2_eval.csv || true

# Zip the exports folder
!zip -r {archive_path} /content/exports

### 6. Programmatic Download to Local PC (No Colab Web UI Needed)
Choose either Method A (1-line curl direct transfer) or Method B (Google Drive sync) below:

In [ ]:
# ==========================================================================
# METHOD A (Recommended for VS Code): Direct CLI Download Link
# Uploads archive to transfer.sh and generates a 1-line curl command to paste
# directly into your local Linux terminal.
# ==========================================================================
import subprocess

archive_path = "/content/onion_grading_colab_artifacts.zip"

print("Uploading package for direct CLI download...")
cmd = f"curl --upload-file {archive_path} https://transfer.sh/onion_grading_colab_artifacts.zip"
res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
download_url = res.stdout.strip()

print("=" * 75)
print("✅ UPLOAD COMPLETE!")
print("Run the following command in your local Linux terminal to download:")
print(f"\n  curl -o ~/Downloads/onion_grading_colab_artifacts.zip {download_url}\n")
print("=" * 75)

In [ ]:
# ==========================================================================
# METHOD B (Alternative): Google Drive Sync
# Mounts your Drive and copies the archive directly to MyDrive
# ==========================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    target_drive_dir = '/content/drive/MyDrive/onion_grading_exports'
    !mkdir -p {target_drive_dir}
    !cp /content/onion_grading_colab_artifacts.zip {target_drive_dir}/
    print(f"\n✅ Saved to Google Drive at: {target_drive_dir}/onion_grading_colab_artifacts.zip")
except Exception as e:
    print(f"Drive mount skipped or error: {e}")